# Soccer Video Analysis on Google Colab

This notebook runs full soccer video analysis using the sports project.

Steps included:
- Setup and install dependencies
- Download pretrained models and sample videos
- Optional upload of your own video
- Run analysis in selected mode
- Preview and download result video

In [ ]:
# Runtime check CELL 1
!nvidia-smi

In [ ]:
# CELL 2      ضبط خوادم DNS يدويًا لتتمكن من الوصول لـ GitHub و Google
!echo "nameserver 8.8.8.8" > /etc/resolv.conf
!echo "nameserver 1.1.1.1" >> /etc/resolv.conf

# الآن جرب تحديث المستودعات (إذا نجح هذا الأمر، فالمشكلة حُلت)
!apt-get update

In [ ]:
#CELL 3 
!ln -s /workspace /content

In [ ]:
#CELL 4      Clone repository and install dependencies
#%cd /workspace
#!git clone https://github.com/roboflow/sports.git sports_repo
%cd /workspace/sports_repo
!pip install -q -U pip
!pip install -q -e ".[soccer]"
!pip install -q -r examples/soccer/requirements.txt

In [ ]:
#CELL5   
!pip install umap-learn scikit-learn tqdm

In [ ]:
# CELL 6    Download pretrained models and sample videos
%cd /workspace/sports_repo/examples/soccer
!bash setup.sh

In [ ]:
# 1. إزالة النسخة الحالية وتثبيت النسخة المستقرة (أقل من 2.0)
!pip install --force-reinstall "numpy<2.0"

In [ ]:
# تحديث supervision لإصدار يدعم VertexLabelAnnotator مع حماية numpy
!pip install -U "supervision>=0.21.0" "numpy<2.0" "scipy<1.13"

In [ ]:
#CELL   8              Configuration (Optimized for RTX 4000 Ada - 20GB VRAM)
from tqdm.auto import tqdm

def ensure_notebook_config(show_progress=True):
    global MODE, MY_TEAM_ID, MY_TEAM_OWN_GOAL_SIDE, VIDEO_FILENAME, PERF_PROFILE
    global FORCE_CUDA, REPO_ROOT, DATA_DIR, PLAYER_DET_IMGSZ, BALL_DET_IMGSZ
    global TEAM_FIT_STRIDE, TRACK_EMBED_SAMPLES, TEAM_REFRESH_EVERY
    global STATS_FRAME_STRIDE, INFER_BATCH_SIZE, SOURCE_VIDEO_PATH, TARGET_VIDEO_PATH
    global DEVICE, USE_FP16

    progress = tqdm(total=8, desc="Cell 8: Configuration", unit="stage") if show_progress else None

    def _step(label):
        if progress is not None:
            progress.set_postfix_str(label)
            progress.update(1)

    # --- إعدادات التوربو ---
    MODE = str(globals().get("MODE", "TEAM_CLASSIFICATION")).strip().upper()
    VIDEO_FILENAME = str(globals().get("VIDEO_FILENAME", "2.mp4")).strip()
    MY_TEAM_ID = int(globals().get("MY_TEAM_ID", 1))
    MY_TEAM_OWN_GOAL_SIDE = str(globals().get("MY_TEAM_OWN_GOAL_SIDE", "right")).strip().lower()
    if MY_TEAM_ID not in (0, 1):
        raise ValueError("MY_TEAM_ID must be 0 or 1.")
    if MY_TEAM_OWN_GOAL_SIDE not in ("left", "right"):
        raise ValueError("MY_TEAM_OWN_GOAL_SIDE must be 'left' or 'right'.")
    PERF_PROFILE = "turbo_speed"
    FORCE_CUDA = True
    _step("Base params")

    import os
    from pathlib import Path
    import torch
    import cv2
    _step("Imports")

    REPO_ROOT = "/workspace/sports_repo"
    DATA_DIR = os.path.join(REPO_ROOT, "examples", "soccer", "data")
    _step("Paths resolved")

    if PERF_PROFILE == "turbo_speed":
        PLAYER_DET_IMGSZ = 640
        BALL_DET_IMGSZ = 640
        TEAM_FIT_STRIDE = 10
        TRACK_EMBED_SAMPLES = 1
        TEAM_REFRESH_EVERY = 2000
        STATS_FRAME_STRIDE = 3
        INFER_BATCH_SIZE = 32

    _step("Turbo Profile applied")

    SOURCE_VIDEO_PATH = os.path.join(DATA_DIR, VIDEO_FILENAME)
    video_stem = Path(SOURCE_VIDEO_PATH).stem
    TARGET_VIDEO_PATH = os.path.join(DATA_DIR, f"output-{video_stem}.mp4")
    _step("Video paths")

    DEVICE = "cuda"
    USE_FP16 = True

    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")
    _step("GPU Optimized")

    cv2.setNumThreads(16)
    torch.set_num_threads(16)
    _step("CPU threads configured")
    _step("Done")

    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    print(f"✅ Turbo Configuration Ready for: {gpu_name}")
    print(f"🚀 Batch Size: {INFER_BATCH_SIZE} | ImgSz: {PLAYER_DET_IMGSZ} | Stride: {STATS_FRAME_STRIDE}")
    print(f"🎨 Team Fit Stride: {TEAM_FIT_STRIDE}")
    print(f"⚽ MY_TEAM_ID: {MY_TEAM_ID} | Own Goal Side: {MY_TEAM_OWN_GOAL_SIDE}")

ensure_notebook_config()


In [ ]:
# CELL 12 (Merged): Advanced match statistics (temporal confirmation + full export + self-bootstrap)
import os
import json
import csv
import time
import glob
import sys
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import supervision as sv
from tqdm import tqdm
from ultralytics import YOLO
from sports.common.team import TeamClassifier, PlayerReIdentifier
from sports.common.ball import BallTracker
import torch


# ---------------------------
# 0) Self-bootstrap settings
# ---------------------------
def _fallback_repo_root() -> str:
    candidate_roots = [
        globals().get("REPO_ROOT"),
        "/workspace/sports_repo",
        "/workspace/sport",
        "/workspace/sports",
        "/content/sports_repo",
        "/content/sport",
        "/content/sports",
        r"d:\sports",
    ]
    for candidate in candidate_roots:
        if candidate and os.path.exists(os.path.join(candidate, "examples", "soccer", "main.py")):
            return os.path.abspath(candidate)

    current = Path.cwd().resolve()
    for base in [current, *current.parents]:
        probe = base / "examples" / "soccer" / "main.py"
        if probe.exists():
            return str(base)

    raise FileNotFoundError("Could not detect REPO_ROOT automatically. Run setup cells first.")


def _fallback_source_video(repo_root: str) -> str:
    explicit = globals().get("SOURCE_VIDEO_PATH")
    video_filename = str(globals().get("VIDEO_FILENAME", "")).strip()

    # If VIDEO_FILENAME itself is a full/relative path, honor it first.
    if video_filename:
        direct_candidate = os.path.abspath(os.path.expanduser(video_filename))
        if os.path.exists(direct_candidate):
            return direct_candidate

    candidate_dirs = []
    data_dir = globals().get("DATA_DIR")
    if data_dir:
        candidate_dirs.append(str(data_dir))
    if repo_root:
        candidate_dirs.append(os.path.join(repo_root, "examples", "soccer", "data"))

    cwd = str(Path.cwd().resolve())
    candidate_dirs.extend([
        cwd,
        os.path.join(cwd, "data"),
        os.path.join(cwd, "examples", "soccer", "data"),
    ])

    seen = set()
    existing_dirs = []
    for d in candidate_dirs:
        if not d:
            continue
        dn = os.path.abspath(str(d))
        if dn in seen or not os.path.isdir(dn):
            continue
        seen.add(dn)
        existing_dirs.append(dn)

    if video_filename:
        for d in existing_dirs:
            cand = os.path.join(d, video_filename)
            if os.path.exists(cand):
                return cand

    # Use explicit path only when VIDEO_FILENAME is empty,
    # or when it points to the same filename.
    if explicit and os.path.exists(explicit):
        explicit_name = os.path.basename(str(explicit))
        if (not video_filename) or (os.path.normcase(explicit_name) == os.path.normcase(video_filename)):
            return str(explicit)

    # Never silently pick a random video when VIDEO_FILENAME is explicitly set.
    if video_filename:
        return ""

    patterns = []
    for d in existing_dirs:
        patterns.extend([
            os.path.join(d, "*.mp4"),
            os.path.join(d, "*.mov"),
            os.path.join(d, "*.mkv"),
        ])
    for pattern in patterns:
        matches = sorted(glob.glob(pattern))
        if matches:
            return matches[0]

    return ""


if "ensure_notebook_config" in globals():
    ensure_notebook_config(show_progress=False)
    if not globals().get("REPO_ROOT") or not os.path.exists(str(globals().get("REPO_ROOT"))):
        try:
            REPO_ROOT = _fallback_repo_root()
        except Exception:
            REPO_ROOT = str(globals().get("REPO_ROOT", ""))
    requested_video_filename = str(globals().get("VIDEO_FILENAME", "")).strip()
    if requested_video_filename:
        SOURCE_VIDEO_PATH = _fallback_source_video(str(globals().get("REPO_ROOT", "")))
    elif not globals().get("SOURCE_VIDEO_PATH") or not os.path.exists(str(globals().get("SOURCE_VIDEO_PATH"))):
        SOURCE_VIDEO_PATH = _fallback_source_video(str(globals().get("REPO_ROOT", "")))
else:
    REPO_ROOT = _fallback_repo_root()
    DEVICE = globals().get("DEVICE") or ("cuda" if torch.cuda.is_available() else "cpu")
    FORCE_CUDA = bool(globals().get("FORCE_CUDA", torch.cuda.is_available()))
    USE_FP16 = bool(globals().get("USE_FP16", DEVICE == "cuda"))
    STATS_FRAME_STRIDE = max(3, int(globals().get("STATS_FRAME_STRIDE", 3)))
    INFER_BATCH_SIZE = int(globals().get("INFER_BATCH_SIZE", 32))
    PLAYER_DET_IMGSZ = int(globals().get("PLAYER_DET_IMGSZ", 640))
    BALL_DET_IMGSZ = int(globals().get("BALL_DET_IMGSZ", 640))
    TEAM_FIT_STRIDE = int(globals().get("TEAM_FIT_STRIDE", 10))
    TRACK_EMBED_SAMPLES = int(globals().get("TRACK_EMBED_SAMPLES", 1))
    TEAM_REFRESH_EVERY = int(globals().get("TEAM_REFRESH_EVERY", 2000))
    MY_TEAM_ID = int(globals().get("MY_TEAM_ID", 1))
    MY_TEAM_OWN_GOAL_SIDE = str(globals().get("MY_TEAM_OWN_GOAL_SIDE", "right")).strip().lower()
    SOURCE_VIDEO_PATH = _fallback_source_video(REPO_ROOT)

if "soccer_main" not in globals():
    soccer_examples_dir = os.path.join(str(REPO_ROOT), "examples", "soccer")
    main_py = os.path.join(soccer_examples_dir, "main.py")
    if os.path.exists(main_py):
        if REPO_ROOT not in sys.path:
            sys.path.insert(0, REPO_ROOT)
        os.chdir(soccer_examples_dir)
        import main as soccer_main
    else:
        try:
            import main as soccer_main
        except Exception as exc:
            raise FileNotFoundError("Could not locate examples/soccer/main.py. Check REPO_ROOT or rerun setup cells.") from exc

if not SOURCE_VIDEO_PATH or not os.path.exists(SOURCE_VIDEO_PATH):
    _vf = str(globals().get("VIDEO_FILENAME", "")).strip()
    _sp = str(globals().get("SOURCE_VIDEO_PATH", "")).strip()
    _probe_dirs = []
    for _d in [globals().get("DATA_DIR"), os.path.join(str(globals().get("REPO_ROOT", "")), "examples", "soccer", "data"), str(Path.cwd().resolve()), os.path.join(str(Path.cwd().resolve()), "data")]:
        if _d:
            _d = os.path.abspath(str(_d))
            if os.path.isdir(_d) and _d not in _probe_dirs:
                _probe_dirs.append(_d)

    _nearby = []
    for _d in _probe_dirs:
        for _patt in ("*.mp4", "*.mov", "*.mkv"):
            _nearby.extend(sorted(glob.glob(os.path.join(_d, _patt))))

    _nearby = list(dict.fromkeys(_nearby))
    _nearby_preview = ", ".join(_nearby[:5]) if _nearby else "none"
    raise FileNotFoundError(
        f"No source video was detected. VIDEO_FILENAME='{_vf}' | SOURCE_VIDEO_PATH='{_sp}'. "
        f"Nearby videos: {_nearby_preview}. Set SOURCE_VIDEO_PATH to a full path, for example: r'E:/videos/match.mp4'"
    )
print(f"[Source Video] using: {SOURCE_VIDEO_PATH}")


# ---------------------------
# 1) Parameters
# ---------------------------
MIN_OWNER_DIST_PX = int(globals().get("MIN_OWNER_DIST_PX", 230))
MIN_OWNER_DIST_PX = max(170, MIN_OWNER_DIST_PX)
MIN_CONFIRM_FRAMES = int(globals().get("MIN_CONFIRM_FRAMES", 2))
RELEASE_CONFIRM_FRAMES = max(3, MIN_CONFIRM_FRAMES)
MIN_PASS_DIST_PX = int(globals().get("MIN_PASS_DIST_PX", 220))
MIN_PASS_TIME_SEC = float(globals().get("MIN_PASS_TIME_SEC", 0.45))
MIN_GROUND_PASS_DIST_PX = int(globals().get("MIN_GROUND_PASS_DIST_PX", 140))
MIN_GROUND_PASS_GAP_SEC = float(globals().get("MIN_GROUND_PASS_GAP_SEC", 1.2))
LIVE_LOG_INTERVAL_SEC = 60.0
TEN_MINUTES_SEC = 600
MAX_FIT_CROPS = 1500
TEAM_BUFFER_SIZE = 64
MIN_PASS_GAP_SEC = float(globals().get("MIN_PASS_GAP_SEC", 2.2))
SAME_PLAYER_SWITCH_DIST_PX = int(globals().get("SAME_PLAYER_SWITCH_DIST_PX", 65))
SIMPLE_PASS_MIN_GAP_SEC = float(globals().get("SIMPLE_PASS_MIN_GAP_SEC", 1.35))
SIMPLE_PASS_MIN_PLAYER_DIST_PX = int(globals().get("SIMPLE_PASS_MIN_PLAYER_DIST_PX", 85))
MIN_BALL_SPEED_PX_PER_SEC = float(globals().get("MIN_BALL_SPEED_PX_PER_SEC", 80.0))
REID_POSITION_TOLERANCE_PX = float(globals().get("REID_POSITION_TOLERANCE_PX", 140.0))
REID_MAX_FRAMES_LOST = int(globals().get("REID_MAX_FRAMES_LOST", 90))

if MY_TEAM_ID not in (0, 1):
    raise ValueError("MY_TEAM_ID must be 0 or 1.")

my_goal_side = str(MY_TEAM_OWN_GOAL_SIDE).strip().lower()
if my_goal_side not in ("left", "right"):
    raise ValueError("MY_TEAM_OWN_GOAL_SIDE must be 'left' or 'right'.")

opponent_team_id = 1 - MY_TEAM_ID
team_goal_side = {
    MY_TEAM_ID: my_goal_side,
    opponent_team_id: ("right" if my_goal_side == "left" else "left"),
}

video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
fps = float(video_info.fps) if video_info.fps else 25.0
total_frames_raw = int(video_info.total_frames)
duration_sec = total_frames_raw / fps if fps > 0 else 0.0

stats_dir = os.path.join(REPO_ROOT, "examples", "soccer", "data", "stats")
os.makedirs(stats_dir, exist_ok=True)

frame_step = max(3, int(STATS_FRAME_STRIDE))
frame_weight = frame_step
infer_batch = max(1, int(INFER_BATCH_SIZE))
player_imgsz = min(640, int(PLAYER_DET_IMGSZ))
ball_imgsz = min(640, int(BALL_DET_IMGSZ))
seconds_per_sampled_frame = (float(frame_step) / fps) if fps > 0 else 0.0
distance_step_scale = float(frame_step)

# With stride>=3, the ball-player pixel gap increases; enforce a safer owner radius.
if frame_step >= 3:
    MIN_OWNER_DIST_PX = max(MIN_OWNER_DIST_PX, 240)

# Hold the last owner briefly when detections blink to reduce "unknown possession" noise.
OWNER_HOLD_FRAMES = max(2, int(round(0.45 / max(seconds_per_sampled_frame, 1e-6))))
MIN_RECEIVE_HOLD_FRAMES = max(2, int(round(0.30 / max(seconds_per_sampled_frame, 1e-6))))

print("Stats extraction config:")
print("- frame_step            :", frame_step)
print("- infer_batch           :", infer_batch)
print("- player imgsz          :", player_imgsz)
print("- ball imgsz            :", ball_imgsz)
print("- sec/sample frame      :", round(seconds_per_sampled_frame, 4))
print("- distance scale        :", distance_step_scale)
print("- fit stride            :", TEAM_FIT_STRIDE)
print("- embed samples         :", TRACK_EMBED_SAMPLES)
print("- refresh every         :", TEAM_REFRESH_EVERY)
print("- team buffer size      :", TEAM_BUFFER_SIZE)
print("- use fp16              :", USE_FP16)
print("- temporal confirm      :", MIN_CONFIRM_FRAMES)
print("- owner dist px         :", MIN_OWNER_DIST_PX)
print("- aerial pass dist px   :", MIN_PASS_DIST_PX)
print("- aerial pass time s    :", MIN_PASS_TIME_SEC)
print("- ground pass dist px   :", MIN_GROUND_PASS_DIST_PX)
print("- ground pass gap s     :", MIN_GROUND_PASS_GAP_SEC)
print("- min pass gap s        :", MIN_PASS_GAP_SEC)
print("- id switch dist px    :", SAME_PLAYER_SWITCH_DIST_PX)
print("- simple pass min gap s :", SIMPLE_PASS_MIN_GAP_SEC)
print("- simple pass min dist  :", SIMPLE_PASS_MIN_PLAYER_DIST_PX)
print("- ball min speed px/s   :", MIN_BALL_SPEED_PX_PER_SEC)
print("- reid pos tolerance px :", REID_POSITION_TOLERANCE_PX)
print("- reid max frames lost  :", REID_MAX_FRAMES_LOST)
print("- receive hold frames   :", MIN_RECEIVE_HOLD_FRAMES)
print("- live log every s      :", LIVE_LOG_INTERVAL_SEC)


# ---------------------------
# 2) Models
# ---------------------------
player_model = YOLO(soccer_main.PLAYER_DETECTION_MODEL_PATH).to(device=DEVICE)
ball_model = YOLO(soccer_main.BALL_DETECTION_MODEL_PATH).to(device=DEVICE)


def _batched(iterable, batch_size):
    batch = []
    for item in iterable:
        batch.append(item)
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:
        yield batch


# ---------------------------
# 3) Fit team classifier
# ---------------------------
fit_crops = []
fit_stride = max(int(TEAM_FIT_STRIDE), frame_step)
fit_generator = sv.get_video_frames_generator(source_path=SOURCE_VIDEO_PATH, stride=fit_stride)
fit_batches = _batched(fit_generator, infer_batch)

for frames_batch in tqdm(fit_batches, desc="collecting crops for team classifier"):
    with torch.inference_mode():
        results = player_model(frames_batch, imgsz=player_imgsz, half=USE_FP16, verbose=False)
    for frame, result in zip(frames_batch, results):
        det = sv.Detections.from_ultralytics(result)
        players_only = det[det.class_id == soccer_main.PLAYER_CLASS_ID]
        fit_crops.extend(soccer_main.get_crops(frame, players_only))
        if len(fit_crops) >= MAX_FIT_CROPS:
            break
    if len(fit_crops) >= MAX_FIT_CROPS:
        break

if len(fit_crops) < 20:
    raise ValueError("Not enough player crops to fit team classifier. Try clearer/longer video.")

team_classifier = TeamClassifier(device=DEVICE)
team_classifier.fit(fit_crops)


def _crop_mean_bgr(crop):
    if crop is None or getattr(crop, "size", 0) == 0:
        return None
    h, w = crop.shape[:2]
    y1, y2 = int(h * 0.25), int(h * 0.85)
    x1, x2 = int(w * 0.20), int(w * 0.80)
    roi = crop[y1:y2, x1:x2]
    if roi.size == 0:
        roi = crop
    return roi.reshape(-1, 3).mean(axis=0)


def _print_color_calibration_logs(crops, classifier, sample_cap=160):
    print("\n[Color Calibration] Team kit calibration from fitted crops:")
    if len(crops) == 0:
        print("- No crops available for calibration logs.")
        return

    sample_n = min(len(crops), int(sample_cap))
    sample_crops = crops[:sample_n]
    pred = classifier.predict(sample_crops).astype(int)

    per_team = {0: [], 1: []}
    for crop, team_id in zip(sample_crops, pred):
        team_id = int(team_id)
        if team_id not in per_team:
            continue
        mean_bgr = _crop_mean_bgr(crop)
        if mean_bgr is not None:
            per_team[team_id].append(mean_bgr)

    for team_id in (0, 1):
        samples = per_team[team_id]
        if len(samples) == 0:
            print(f"- Team {team_id}: no reliable color sample")
            continue
        avg_bgr = np.mean(np.vstack(samples), axis=0)
        avg_rgb = avg_bgr[::-1]
        print(
            f"- Team {team_id}: samples={len(samples)} | avg_rgb=({int(avg_rgb[0])}, {int(avg_rgb[1])}, {int(avg_rgb[2])})"
        )

    print(f"- MY_TEAM_ID selected: {int(MY_TEAM_ID)}")


_print_color_calibration_logs(fit_crops, team_classifier)


# ---------------------------
# 4) Trackers and helpers
# ---------------------------
player_tracker = sv.ByteTrack(
    track_activation_threshold=0.35,
    lost_track_buffer=200,
    minimum_matching_threshold=0.3,
    minimum_consecutive_frames=3,
)
ball_tracker = BallTracker(buffer_size=20)
player_reid = PlayerReIdentifier(
    max_frames_lost=REID_MAX_FRAMES_LOST,
    position_tolerance_px=REID_POSITION_TOLERANCE_PX,
)


def _detect_ball_batch(frames_batch):
    with torch.inference_mode():
        results = ball_model(frames_batch, imgsz=ball_imgsz, half=USE_FP16, verbose=False)
    return [sv.Detections.from_ultralytics(result).with_nms(threshold=0.1) for result in results]


def _to_track_id(value):
    if value is None:
        return None
    try:
        if np.isnan(value):
            return None
    except Exception:
        pass
    try:
        return int(value)
    except Exception:
        return None


def _safe_goalkeeper_team(players, players_team_id, goalkeepers):
    if len(goalkeepers) == 0:
        return np.array([], dtype=int)
    if len(players) == 0 or len(players_team_id) == 0:
        return np.full(len(goalkeepers), -1, dtype=int)

    valid = players_team_id[np.isin(players_team_id, np.array([0, 1]))]
    if len(valid) == 0:
        return np.full(len(goalkeepers), -1, dtype=int)

    unique_valid = np.unique(valid)
    if len(unique_valid) < 2:
        return np.full(len(goalkeepers), int(unique_valid[0]), dtype=int)

    majority = Counter(valid.tolist()).most_common(1)[0][0]
    tmp_team_id = players_team_id.copy()
    tmp_team_id[tmp_team_id == -1] = int(majority)
    return soccer_main.resolve_goalkeepers_team_id(players, tmp_team_id, goalkeepers).astype(int)


def _base_third_from_x(x, frame_width):
    if x < frame_width / 3.0:
        return "left"
    if x < 2.0 * frame_width / 3.0:
        return "middle"
    return "right"


def _phase_third_for_team(base_third, team_id):
    if base_third == "middle":
        return "middle"
    side = team_goal_side.get(team_id, None)
    if side is None:
        return None
    if side == "left":
        return "defensive" if base_third == "left" else "offensive"
    return "defensive" if base_third == "right" else "offensive"


# ---------------------------
# 5) Stats containers
# ---------------------------
team_possession_frames = Counter()
window_total_frames = Counter()
window_team_frames = defaultdict(Counter)
team_touches = Counter()
team_passes_completed = Counter()
team_turnovers_lost = Counter()
team_interceptions_won = Counter()
team_detected_count_sum = Counter()

team_third_possession_frames = defaultdict(Counter)
window_team_third_frames = defaultdict(lambda: defaultdict(Counter))

player_touches = Counter()
player_possession_frames = Counter()
player_distance_px = Counter()
last_player_xy = {}

track_team_votes = defaultdict(Counter)
track_embedding_samples = Counter()
track_team_cache = {}
team_pending_crops = []
team_pending_meta = []
team_pending_tids = set()
last_pass_release_time_by_team = {0: -1e9, 1: -1e9}
current_owner_last_xy = None
last_owner_before_release = None
last_owner_before_release_xy = None
my_team_overlay_by_frame = defaultdict(list)


def _flush_team_buffer(force=False):
    global team_pending_crops, team_pending_meta, team_pending_tids
    while len(team_pending_crops) >= TEAM_BUFFER_SIZE or (force and team_pending_crops):
        chunk_size = min(len(team_pending_crops), TEAM_BUFFER_SIZE)
        crops_chunk = team_pending_crops[:chunk_size]
        meta_chunk = team_pending_meta[:chunk_size]

        pred_ids = team_classifier.predict(crops_chunk).astype(int)
        for pred_team, meta in zip(pred_ids, meta_chunk):
            tid = meta["tid"]
            if tid is None:
                continue

            pred_team = int(pred_team)
            if pred_team in (0, 1):
                track_team_cache[tid] = pred_team
                track_team_votes[tid][pred_team] += 1
            track_embedding_samples[tid] += 1

        for meta in meta_chunk:
            team_pending_tids.discard(meta["tid"])
        del team_pending_crops[:chunk_size]
        del team_pending_meta[:chunk_size]


events = []
last_ball_xy = None
ball_distance_px = 0.0
last_ball_velocity_px_per_sec = 0.0  # instantaneous speed of ball at last sampled frame

candidate_owner = None
candidate_counter = 0
confirmed_owner = None
no_owner_counter = 0

current_owner = None
last_team_owner = None
ball_release_xy = None
ball_release_frame = 0
last_logged_minute = 0


def _register_simple_pass(team_id, event_time_sec, distance_px, flight_time_sec, pass_kind="simple_same_team_transfer", ball_speed_px_per_sec=0.0):
    team_id = int(team_id)
    distance_px = float(distance_px)
    event_time_sec = float(event_time_sec)
    ball_speed_px_per_sec = float(ball_speed_px_per_sec)

    # Debounce very-close pass events (usually dribble touches or noisy ownership flicker).
    prev_time = float(last_pass_release_time_by_team.get(team_id, -1e9))
    if (event_time_sec - prev_time) < SIMPLE_PASS_MIN_GAP_SEC:
        return False

    # Ignore tiny owner changes that are likely not real passes.
    if distance_px < float(SIMPLE_PASS_MIN_PLAYER_DIST_PX):
        return False

    # Ignore events where the ball was barely moving (not a real pass).
    if ball_speed_px_per_sec > 0 and ball_speed_px_per_sec < MIN_BALL_SPEED_PX_PER_SEC:
        return False

    team_passes_completed[team_id] += 1
    last_pass_release_time_by_team[team_id] = event_time_sec
    events.append(
        {
            "time_sec": round(event_time_sec, 2),
            "event": "team_pass",
            "team": team_id,
            "distance_px": round(distance_px, 1),
            "flight_time_sec": round(float(flight_time_sec), 2),
            "pass_kind": str(pass_kind),
            "ball_speed_px_per_sec": round(ball_speed_px_per_sec, 1),
        }
    )
    return True


# ---------------------------
# 6) Main loop
# ---------------------------
raw_frame_generator = sv.get_video_frames_generator(source_path=SOURCE_VIDEO_PATH, stride=frame_step)
num_processed_frames = int(np.ceil(total_frames_raw / frame_step))
num_batches = int(np.ceil(num_processed_frames / infer_batch))

processed_count = 0
t0 = time.time()

for frames_batch in tqdm(_batched(raw_frame_generator, infer_batch), total=num_batches, desc="extracting stats (batches)"):
    with torch.inference_mode():
        player_results = player_model(frames_batch, imgsz=player_imgsz, half=USE_FP16, verbose=False)
        ball_results = _detect_ball_batch(frames_batch)

    batch_records = []

    for local_i, frame in enumerate(frames_batch):
        frame_idx = (processed_count + local_i) * frame_step
        player_result = player_results[local_i]

        detections = sv.Detections.from_ultralytics(player_result)
        detections = player_tracker.update_with_detections(detections)

        players = detections[detections.class_id == soccer_main.PLAYER_CLASS_ID]
        goalkeepers = detections[detections.class_id == soccer_main.GOALKEEPER_CLASS_ID]

        players_xy = players.get_anchors_coordinates(sv.Position.BOTTOM_CENTER) if len(players) > 0 else np.empty((0, 2))
        players_tid_raw = players.tracker_id if players.tracker_id is not None else np.array([None] * len(players))
        player_crops = soccer_main.get_crops(frame, players) if len(players) > 0 else []

        if len(players) > 0:
            for i in range(len(players)):
                tid = _to_track_id(players_tid_raw[i])
                if tid is None or i >= len(player_crops):
                    continue
                if tid in track_team_cache or tid in team_pending_tids:
                    continue
                should_refresh = (track_embedding_samples[tid] == 0) or (frame_idx % TEAM_REFRESH_EVERY == 0)
                if should_refresh and track_embedding_samples[tid] < TRACK_EMBED_SAMPLES:
                    team_pending_crops.append(player_crops[i])
                    team_pending_meta.append({"tid": tid})
                    team_pending_tids.add(tid)

        ball_det = ball_tracker.update(ball_results[local_i])

        batch_records.append(
            {
                "frame": frame,
                "frame_idx": frame_idx,
                "players": players,
                "goalkeepers": goalkeepers,
                "players_xy": players_xy,
                "players_tid_raw": players_tid_raw,
                "ball_det": ball_det,
            }
        )

    _flush_team_buffer(force=False)

    for record in batch_records:
        frame = record["frame"]
        frame_idx = record["frame_idx"]
        players = record["players"]
        goalkeepers = record["goalkeepers"]
        players_xy = record["players_xy"]
        players_tid_raw = record["players_tid_raw"]
        ball_det = record["ball_det"]

        owner_candidates = []
        players_team_id = np.array([], dtype=int)

        if len(players) > 0:
            raw_tids = [_to_track_id(players_tid_raw[i]) for i in range(len(players))]
            # Re-map raw ByteTrack IDs to stable canonical IDs using position-based re-ID.
            norm_tids = [
                player_reid.get_stable_id(raw_tids[i], players_xy[i])
                if raw_tids[i] is not None else None
                for i in range(len(players))
            ]
            resolved_team_ids = []
            for i in range(len(players)):
                tid = norm_tids[i]
                if tid is None:
                    resolved_team_ids.append(-1)
                elif tid in track_team_cache:
                    resolved_team_ids.append(int(track_team_cache[tid]))
                elif len(track_team_votes[tid]) > 0:
                    resolved_team_ids.append(int(track_team_votes[tid].most_common(1)[0][0]))
                else:
                    resolved_team_ids.append(-1)

                xy = players_xy[i]
                if tid is not None:
                    prev_xy = last_player_xy.get(tid)
                    if prev_xy is not None:
                        player_distance_px[tid] += float(np.linalg.norm(xy - prev_xy)) * distance_step_scale
                    last_player_xy[tid] = xy

            # Fallback: classify unresolved field players immediately in this frame
            # so overlay does not wait for TEAM_BUFFER_SIZE to fill.
            unresolved_idx = [
                i for i, team_id in enumerate(resolved_team_ids)
                if team_id not in (0, 1) and i < len(player_crops)
            ]
            if len(unresolved_idx) > 0:
                fallback_crops = [player_crops[i] for i in unresolved_idx]
                try:
                    fallback_pred = team_classifier.predict(fallback_crops).astype(int)
                except Exception:
                    fallback_pred = np.array([], dtype=int)

                for idx_local, pred_team in zip(unresolved_idx, fallback_pred):
                    pred_team = int(pred_team)
                    if pred_team not in (0, 1):
                        continue

                    resolved_team_ids[idx_local] = pred_team
                    tid = norm_tids[idx_local]
                    if tid is not None:
                        track_team_cache[tid] = pred_team
                        track_team_votes[tid][pred_team] += 1
                        track_embedding_samples[tid] += 1
                        # Propagate team assignment into the ReID gallery.
                        player_reid.get_stable_id(tid, players_xy[idx_local], team_id=pred_team)

            for i in range(len(players)):
                owner_candidates.append({"tid": norm_tids[i], "team": resolved_team_ids[i], "xy": players_xy[i]})

            players_team_id = np.array(resolved_team_ids, dtype=int)

            player_xyxy = players.xyxy if getattr(players, "xyxy", None) is not None else None
            if player_xyxy is not None:
                frame_overlay = my_team_overlay_by_frame[int(frame_idx)]
                for i in range(len(players)):
                    tid = norm_tids[i]
                    team_id = resolved_team_ids[i]
                    if team_id != MY_TEAM_ID or tid is None or i >= len(player_xyxy):
                        continue

                    x1, y1, x2, y2 = player_xyxy[i]
                    x_center = int(round((float(x1) + float(x2)) * 0.5))
                    y_feet = int(round(float(y2)))
                    rx = max(10, int(round((float(x2) - float(x1)) * 0.32)))
                    ry = max(6, int(round((float(y2) - float(y1)) * 0.12)))

                    frame_overlay.append(
                        {
                            "tracker_id": int(tid),
                            "team_id": int(team_id),
                            "x": x_center,
                            "y": y_feet,
                            "rx": rx,
                            "ry": ry,
                        }
                    )

        if len(goalkeepers) > 0:
            gk_xy = goalkeepers.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
            gk_tid_raw = goalkeepers.tracker_id if goalkeepers.tracker_id is not None else np.array([None] * len(goalkeepers))
            goalkeepers_team_id = _safe_goalkeeper_team(players, players_team_id, goalkeepers)
            # Apply stable-ID re-mapping for goalkeepers.
            gk_tids_stable = [
                player_reid.get_stable_id(_to_track_id(gk_tid_raw[i]), gk_xy[i])
                if _to_track_id(gk_tid_raw[i]) is not None else None
                for i in range(len(goalkeepers))
            ]

            for i in range(len(goalkeepers)):
                tid = gk_tids_stable[i]
                team_id = int(goalkeepers_team_id[i]) if i < len(goalkeepers_team_id) else -1
                xy = gk_xy[i]

                if tid is not None:
                    if team_id in (0, 1):
                        track_team_votes[tid][team_id] += 1
                        track_team_cache[tid] = team_id
                    prev_xy = last_player_xy.get(tid)
                    if prev_xy is not None:
                        player_distance_px[tid] += float(np.linalg.norm(xy - prev_xy)) * distance_step_scale
                    last_player_xy[tid] = xy

                owner_candidates.append({"tid": tid, "team": team_id, "xy": xy})

                if team_id == MY_TEAM_ID and tid is not None:
                    frame_overlay = my_team_overlay_by_frame[int(frame_idx)]
                    x_center = int(round(float(xy[0])))
                    y_feet = int(round(float(xy[1])))
                    frame_overlay.append(
                        {
                            "tracker_id": int(tid),
                            "team_id": int(team_id),
                            "x": x_center,
                            "y": y_feet,
                            "rx": 12,
                            "ry": 8,
                        }
                    )

        owner_xy_by_key = {}
        for candidate in owner_candidates:
            if candidate["team"] in (0, 1):
                owner_xy_by_key[(candidate["tid"], candidate["team"])] = candidate["xy"]

        frame_team_counts = Counter([c["team"] for c in owner_candidates if c["team"] in (0, 1)])
        for t in (0, 1):
            team_detected_count_sum[t] += frame_team_counts[t] * frame_weight

        ball_xy = None
        if len(ball_det) > 0:
            ball_xy = ball_det.get_anchors_coordinates(sv.Position.CENTER)[0]
            if last_ball_xy is not None:
                ball_step_dist = float(np.linalg.norm(ball_xy - last_ball_xy)) * distance_step_scale
                ball_distance_px += ball_step_dist
                # Track instantaneous ball speed (px/s) for pass validation.
                last_ball_velocity_px_per_sec = (
                    float(np.linalg.norm(ball_xy - last_ball_xy)) / max(seconds_per_sampled_frame, 1e-6)
                )
            last_ball_xy = ball_xy
        elif last_ball_xy is not None:
            # Fallback to the latest known ball position when detector misses for a short span.
            ball_xy = np.array(last_ball_xy, dtype=float)

        current_nearest = None
        if ball_xy is not None and len(owner_candidates) > 0:
            dists = [float(np.linalg.norm(candidate["xy"] - ball_xy)) for candidate in owner_candidates]
            best = int(np.argmin(dists))
            distance_to_ball = dists[best]

            if distance_to_ball <= MIN_OWNER_DIST_PX:
                nearest = owner_candidates[best]
                tid = nearest["tid"]
                team = nearest["team"]

                if team in (0, 1):
                    current_nearest = (tid, team)

        # Temporal hold to smooth short detector dropouts and avoid inflated unknown possession.
        if current_nearest is None and current_owner is not None and no_owner_counter < OWNER_HOLD_FRAMES:
            current_nearest = current_owner

        if current_nearest is None:
            no_owner_counter += 1
            candidate_owner = None
            candidate_counter = 0
            if no_owner_counter >= RELEASE_CONFIRM_FRAMES:
                confirmed_owner = None
        else:
            no_owner_counter = 0
            if current_nearest == candidate_owner:
                candidate_counter += 1
            else:
                candidate_owner = current_nearest
                candidate_counter = 1

            if candidate_counter >= MIN_CONFIRM_FRAMES:
                confirmed_owner = candidate_owner

        owner_candidate = confirmed_owner
        owner_candidate_xy = owner_xy_by_key.get(owner_candidate) if owner_candidate is not None else None

        time_sec = frame_idx / fps if fps > 0 else 0.0
        window_idx = int(time_sec // TEN_MINUTES_SEC)
        window_total_frames[window_idx] += frame_weight

        if owner_candidate is not None:
            owner_tid, owner_team = owner_candidate
            # Weight possession by proximity: closer ball → more confident possession.
            # Clamp to [0.5, 1.0] so distant ownership still counts, just less.
            if ball_xy is not None and owner_candidate_xy is not None:
                _dist_ratio = min(1.0, float(np.linalg.norm(ball_xy - owner_candidate_xy)) / max(1.0, float(MIN_OWNER_DIST_PX)))
                _poss_weight = frame_weight * max(0.5, 1.0 - 0.5 * _dist_ratio)
            else:
                _poss_weight = frame_weight
            team_possession_frames[owner_team] += _poss_weight
            window_team_frames[window_idx][owner_team] += _poss_weight
            if owner_tid is not None:
                player_possession_frames[owner_tid] += _poss_weight

            if ball_xy is not None:
                base_third = _base_third_from_x(float(ball_xy[0]), frame.shape[1])
                phase_third = _phase_third_for_team(base_third, owner_team)
                if phase_third is not None:
                    team_third_possession_frames[owner_team][phase_third] += frame_weight
                    window_team_third_frames[window_idx][owner_team][phase_third] += frame_weight

        previous_owner = current_owner
        previous_owner_xy = current_owner_last_xy

        if owner_candidate != current_owner:
            if current_owner is not None and owner_candidate is None:
                last_owner_before_release = current_owner
                last_owner_before_release_xy = current_owner_last_xy
                last_team_owner = current_owner[1]
                ball_release_xy = None if ball_xy is None else np.array(ball_xy, dtype=float)
                ball_release_frame = frame_idx

            if owner_candidate is not None:
                new_tid, new_team = owner_candidate
                team_touches[new_team] += 1
                if new_tid is not None:
                    player_touches[new_tid] += 1

                if last_team_owner is not None:
                    release_owner_tid = last_owner_before_release[0] if last_owner_before_release is not None else None
                    release_owner_team = last_owner_before_release[1] if last_owner_before_release is not None else last_team_owner

                    if new_team == last_team_owner:
                        same_player_reclaim = (
                            release_owner_tid is not None
                            and new_tid is not None
                            and int(new_tid) == int(release_owner_tid)
                        )
                        if (
                            not same_player_reclaim
                            and last_owner_before_release_xy is not None
                            and owner_candidate_xy is not None
                        ):
                            same_player_reclaim = (
                                float(np.linalg.norm(owner_candidate_xy - last_owner_before_release_xy))
                                <= SAME_PLAYER_SWITCH_DIST_PX
                            )

                        if not same_player_reclaim:
                            dist_traveled = 0.0
                            if ball_release_xy is not None and ball_xy is not None:
                                dist_traveled = float(np.linalg.norm(ball_xy - ball_release_xy))
                            if last_owner_before_release_xy is not None and owner_candidate_xy is not None:
                                dist_traveled = max(
                                    dist_traveled,
                                    float(np.linalg.norm(owner_candidate_xy - last_owner_before_release_xy)),
                                )

                            flight_time = ((frame_idx - ball_release_frame) / fps) if fps > 0 else 0.0
                            release_time_sec = (ball_release_frame / fps) if fps > 0 else 0.0

                            # Simple rule: same-team transfer between different players = pass.
                            _register_simple_pass(
                                new_team,
                                release_time_sec,
                                dist_traveled,
                                flight_time,
                                pass_kind="simple_same_team_transfer",
                                ball_speed_px_per_sec=last_ball_velocity_px_per_sec,
                            )
                    else:
                        team_turnovers_lost[release_owner_team] += 1
                        team_interceptions_won[new_team] += 1
                        turnover_dist = 0.0
                        if ball_release_xy is not None and owner_candidate_xy is not None:
                            turnover_dist = float(np.linalg.norm(owner_candidate_xy - ball_release_xy))
                        events.append(
                            {
                                "time_sec": round(time_sec, 2),
                                "event": "team_interception",
                                "lost_by_team": int(release_owner_team),
                                "won_by_team": int(new_team),
                                "distance_px": round(turnover_dist, 1),
                            }
                        )

                elif previous_owner is not None:
                    prev_tid, prev_team = previous_owner
                    if (
                        prev_team == new_team
                        and prev_tid is not None
                        and new_tid is not None
                        and prev_tid != new_tid
                        and previous_owner_xy is not None
                        and owner_candidate_xy is not None
                    ):
                        direct_dist = float(np.linalg.norm(owner_candidate_xy - previous_owner_xy))
                        direct_release_frame = max(frame_idx - frame_step, 0)
                        direct_release_time_sec = (direct_release_frame / fps) if fps > 0 else 0.0
                        likely_id_switch = direct_dist <= SAME_PLAYER_SWITCH_DIST_PX

                        if not likely_id_switch:
                            _register_simple_pass(
                                new_team,
                                direct_release_time_sec,
                                direct_dist,
                                seconds_per_sampled_frame,
                                pass_kind="simple_same_team_transfer",
                                ball_speed_px_per_sec=last_ball_velocity_px_per_sec,
                            )
                    elif prev_team != new_team:
                        team_turnovers_lost[prev_team] += 1
                        team_interceptions_won[new_team] += 1
                        turnover_dist = 0.0
                        if previous_owner_xy is not None and owner_candidate_xy is not None:
                            turnover_dist = float(np.linalg.norm(owner_candidate_xy - previous_owner_xy))
                        events.append(
                            {
                                "time_sec": round(time_sec, 2),
                                "event": "team_interception",
                                "lost_by_team": int(prev_team),
                                "won_by_team": int(new_team),
                                "distance_px": round(turnover_dist, 1),
                            }
                        )

                current_owner = owner_candidate
                last_team_owner = None
                last_owner_before_release = None
                last_owner_before_release_xy = None
                ball_release_xy = None
                ball_release_frame = 0
            else:
                current_owner = None
                current_owner_last_xy = None

        if owner_candidate is not None and owner_candidate_xy is not None:
            current_owner_last_xy = owner_candidate_xy

        # Advance ReID gallery age; prune tracks that have been lost too long.
        player_reid.end_frame()

        current_minute = int(time_sec // LIVE_LOG_INTERVAL_SEC)
        if current_minute > last_logged_minute:
            print(
                f"[Live Counter] الدقيقة {current_minute}: "
                f"Team 0 = {int(team_passes_completed[0])} | "
                f"Team 1 = {int(team_passes_completed[1])}"
            )
            last_logged_minute = current_minute

    processed_count += len(frames_batch)

_flush_team_buffer(force=True)
my_team_overlay_tracks = {int(frame_k): rows for frame_k, rows in my_team_overlay_by_frame.items() if len(rows) > 0}
globals()["my_team_overlay_tracks"] = my_team_overlay_tracks
print(f"[Overlay] my-team annotated sampled frames: {len(my_team_overlay_tracks)}")

if torch.cuda.is_available():
    torch.cuda.synchronize()
dt_stats = max(time.time() - t0, 1e-6)
print(f"Stats cell effective throughput: {num_processed_frames / dt_stats:.2f} fps")
print(
    f"[Live Counter] النهائي: Team 0 = {int(team_passes_completed[0])} | "
    f"Team 1 = {int(team_passes_completed[1])}"
)


# ---------------------------
# 7) Final aggregation
# ---------------------------
track_team_final = {}
for tid, votes in track_team_votes.items():
    if len(votes) > 0:
        track_team_final[tid] = votes.most_common(1)[0][0]

team_distance_px = Counter()
for tid, dist in player_distance_px.items():
    team_id = track_team_final.get(tid, None)
    if team_id in (0, 1):
        team_distance_px[team_id] += float(dist)


def _pct(n, d):
    return 100.0 * float(n) / float(d) if d else 0.0


total_frames_est = int(sum(window_total_frames.values()))
team_0_frames = int(team_possession_frames[0])
team_1_frames = int(team_possession_frames[1])
unknown_frames = max(total_frames_est - team_0_frames - team_1_frames, 0)
known_frames = team_0_frames + team_1_frames

team_stats_rows = []
for team_id in [0, 1]:
    passes_completed = team_passes_completed[team_id]
    pass_attempts_est = passes_completed + team_turnovers_lost[team_id]
    team_stats_rows.append(
        {
            "team_id": team_id,
            "possession_pct_total_frames": round(_pct(team_possession_frames[team_id], total_frames_est), 2),
            "possession_pct_when_known": round(_pct(team_possession_frames[team_id], known_frames), 2),
            "touches": int(team_touches[team_id]),
            "passes_completed_est": int(passes_completed),
            "pass_attempts_est": int(pass_attempts_est),
            "pass_accuracy_est_pct": round(_pct(passes_completed, pass_attempts_est), 2),
            "turnovers_lost": int(team_turnovers_lost[team_id]),
            "interceptions_won": int(team_interceptions_won[team_id]),
            "distance_covered_px": round(float(team_distance_px[team_id]), 2),
            "avg_detected_players_per_frame": round(float(team_detected_count_sum[team_id]) / float(total_frames_est if total_frames_est else 1), 2),
        }
    )

window_rows = []
for w in sorted(window_total_frames.keys()):
    w_total = int(window_total_frames[w])
    w_t0 = int(window_team_frames[w][0])
    w_t1 = int(window_team_frames[w][1])
    w_unknown = max(w_total - w_t0 - w_t1, 0)
    start_sec = int(w * TEN_MINUTES_SEC)
    end_sec = int(min(duration_sec, (w + 1) * TEN_MINUTES_SEC))
    window_rows.append(
        {
            "window_index": int(w),
            "start_sec": start_sec,
            "end_sec": end_sec,
            "team_0_possession_pct": round(_pct(w_t0, w_total), 2),
            "team_1_possession_pct": round(_pct(w_t1, w_total), 2),
            "unknown_pct": round(_pct(w_unknown, w_total), 2),
            "frames": w_total,
        }
    )

third_labels = ["defensive", "middle", "offensive"]
team_thirds_rows = []
for team_id in [0, 1]:
    team_frames = sum(team_third_possession_frames[team_id][k] for k in third_labels)
    row = {
        "team_id": int(team_id),
        "goal_side": team_goal_side.get(team_id, "unknown"),
        "defensive_frames": int(team_third_possession_frames[team_id]["defensive"]),
        "middle_frames": int(team_third_possession_frames[team_id]["middle"]),
        "offensive_frames": int(team_third_possession_frames[team_id]["offensive"]),
        "defensive_sec": round(team_third_possession_frames[team_id]["defensive"] / fps if fps > 0 else 0.0, 2),
        "middle_sec": round(team_third_possession_frames[team_id]["middle"] / fps if fps > 0 else 0.0, 2),
        "offensive_sec": round(team_third_possession_frames[team_id]["offensive"] / fps if fps > 0 else 0.0, 2),
        "defensive_pct_of_team_possession": round(_pct(team_third_possession_frames[team_id]["defensive"], team_frames), 2),
        "middle_pct_of_team_possession": round(_pct(team_third_possession_frames[team_id]["middle"], team_frames), 2),
        "offensive_pct_of_team_possession": round(_pct(team_third_possession_frames[team_id]["offensive"], team_frames), 2),
        "defensive_pct_of_total_frames": round(_pct(team_third_possession_frames[team_id]["defensive"], total_frames_est), 2),
        "middle_pct_of_total_frames": round(_pct(team_third_possession_frames[team_id]["middle"], total_frames_est), 2),
        "offensive_pct_of_total_frames": round(_pct(team_third_possession_frames[team_id]["offensive"], total_frames_est), 2),
    }
    team_thirds_rows.append(row)

window_thirds_rows = []
for w in sorted(window_total_frames.keys()):
    start_sec = int(w * TEN_MINUTES_SEC)
    end_sec = int(min(duration_sec, (w + 1) * TEN_MINUTES_SEC))
    for team_id in [0, 1]:
        def_f = int(window_team_third_frames[w][team_id]["defensive"])
        mid_f = int(window_team_third_frames[w][team_id]["middle"])
        off_f = int(window_team_third_frames[w][team_id]["offensive"])
        team_w_frames = def_f + mid_f + off_f
        window_thirds_rows.append(
            {
                "window_index": int(w),
                "start_sec": start_sec,
                "end_sec": end_sec,
                "team_id": int(team_id),
                "goal_side": team_goal_side.get(team_id, "unknown"),
                "defensive_frames": def_f,
                "middle_frames": mid_f,
                "offensive_frames": off_f,
                "defensive_sec": round(def_f / fps if fps > 0 else 0.0, 2),
                "middle_sec": round(mid_f / fps if fps > 0 else 0.0, 2),
                "offensive_sec": round(off_f / fps if fps > 0 else 0.0, 2),
                "defensive_pct_of_team_window": round(_pct(def_f, team_w_frames), 2),
                "middle_pct_of_team_window": round(_pct(mid_f, team_w_frames), 2),
                "offensive_pct_of_team_window": round(_pct(off_f, team_w_frames), 2),
            }
        )

player_rows = []
all_player_ids = set(player_touches.keys()) | set(player_possession_frames.keys()) | set(player_distance_px.keys())
for tid in sorted(all_player_ids):
    team_id = track_team_final.get(tid, -1)
    player_rows.append(
        {
            "tracker_id": int(tid),
            "team_id": int(team_id),
            "touches": int(player_touches[tid]),
            "possession_seconds": round((float(player_possession_frames[tid]) / float(frame_step)) * seconds_per_sampled_frame if fps > 0 else 0.0, 2),
            "distance_covered_px": round(float(player_distance_px[tid]), 2),
        }
    )

overlay_rows = []
for frame_k in sorted(my_team_overlay_tracks.keys()):
    for row in my_team_overlay_tracks[frame_k]:
        overlay_rows.append(
            {
                "frame_idx": int(frame_k),
                "tracker_id": int(row["tracker_id"]),
                "team_id": int(row["team_id"]),
                "x": int(row["x"]),
                "y": int(row["y"]),
                "rx": int(row["rx"]),
                "ry": int(row["ry"]),
            }
        )

my_team_thirds = {
    "team_id": int(MY_TEAM_ID),
    "goal_side": team_goal_side[MY_TEAM_ID],
    "defensive_sec": round(team_third_possession_frames[MY_TEAM_ID]["defensive"] / fps if fps > 0 else 0.0, 2),
    "middle_sec": round(team_third_possession_frames[MY_TEAM_ID]["middle"] / fps if fps > 0 else 0.0, 2),
    "offensive_sec": round(team_third_possession_frames[MY_TEAM_ID]["offensive"] / fps if fps > 0 else 0.0, 2),
}

summary = {
    "source_video_path": SOURCE_VIDEO_PATH,
    "fps": round(fps, 3),
    "frame_step": int(frame_step),
    "infer_batch": int(infer_batch),
    "total_frames_original": int(total_frames_raw),
    "total_frames_estimated": int(total_frames_est),
    "duration_sec": round(duration_sec, 2),
    "duration_min": round(duration_sec / 60.0, 2),
    "ball_distance_px": round(ball_distance_px, 2),
    "ball_avg_speed_px_per_sec": round((ball_distance_px / duration_sec) if duration_sec > 0 else 0.0, 2),
    "my_team": {
        "team_id": int(MY_TEAM_ID),
        "own_goal_side": my_goal_side,
    },
    "possession": {
        "team_0_pct_total_frames": round(_pct(team_0_frames, total_frames_est), 2),
        "team_1_pct_total_frames": round(_pct(team_1_frames, total_frames_est), 2),
        "unknown_pct_total_frames": round(_pct(unknown_frames, total_frames_est), 2),
        "team_0_pct_when_known": round(_pct(team_0_frames, known_frames), 2),
        "team_1_pct_when_known": round(_pct(team_1_frames, known_frames), 2),
    },
    "thirds_possession": {
        "team_0": team_thirds_rows[0],
        "team_1": team_thirds_rows[1],
        "my_team": my_team_thirds,
    },
    "notes": [
        "Temporal confirmation stabilizes possession labels.",
        "Possession frames are weighted by ball-to-player proximity (0.5-1.0 scale).",
        "Pass events require ball speed >= MIN_BALL_SPEED_PX_PER_SEC to filter out dribble noise.",
        "PlayerReIdentifier re-maps short-lived ByteTrack IDs to stable canonical IDs.",
        f"ReID: position_tolerance={REID_POSITION_TOLERANCE_PX:.0f}px, max_frames_lost={REID_MAX_FRAMES_LOST}.",
        "Possession accuracy: reported both as % of total frames and % of known-possession frames.",
        "Batched player inference increases GPU utilization.",
        "Track cache reduces repeated embedding extraction.",
        f"Team buffer size={TEAM_BUFFER_SIZE} batches unknown player classifications.",
        "Passing/interception stats use team-level ball displacement, not tracker-ID handoff.",
        "Release-to-reception path now counts only same-team passes, not opponent receptions.",
        f"ID switch guard: same-team owner changes under {SAME_PLAYER_SWITCH_DIST_PX:.0f}px are treated as the same player.",
        f"Pass debounce: ignore same-team passes closer than {SIMPLE_PASS_MIN_GAP_SEC:.2f}s.",
        f"Pass min distance: ignore same-team passes shorter than {SIMPLE_PASS_MIN_PLAYER_DIST_PX:.0f}px.",
        "Renderer now filters overlay circles by MY_TEAM_ID only.",
        "Pass rule: same-team transfer between different players is counted directly (ground or air).",
        "ByteTrack thresholds were tightened to reduce tracker ID explosion.",
    ],
}


# ---------------------------
# 8) Save outputs
# ---------------------------
print("Saving statistics files...")
summary_json_path = os.path.join(stats_dir, "summary.json")
team_stats_csv_path = os.path.join(stats_dir, "team_stats.csv")
player_stats_csv_path = os.path.join(stats_dir, "player_stats.csv")
possession_10min_csv_path = os.path.join(stats_dir, "possession_10min.csv")
events_csv_path = os.path.join(stats_dir, "events.csv")
team_thirds_csv_path = os.path.join(stats_dir, "team_possession_by_third.csv")
thirds_10min_csv_path = os.path.join(stats_dir, "possession_by_third_10min.csv")
my_team_thirds_json_path = os.path.join(stats_dir, "my_team_thirds_summary.json")
my_team_overlay_csv_path = os.path.join(stats_dir, "my_team_overlay_tracks.csv")

with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

with open(my_team_thirds_json_path, "w", encoding="utf-8") as f:
    json.dump(my_team_thirds, f, ensure_ascii=False, indent=2)

if len(team_stats_rows) > 0:
    with open(team_stats_csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(team_stats_rows[0].keys()))
        writer.writeheader()
        writer.writerows(team_stats_rows)

if len(team_thirds_rows) > 0:
    with open(team_thirds_csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(team_thirds_rows[0].keys()))
        writer.writeheader()
        writer.writerows(team_thirds_rows)

with open(player_stats_csv_path, "w", newline="", encoding="utf-8") as f:
    fieldnames = ["tracker_id", "team_id", "touches", "possession_seconds", "distance_covered_px"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(player_rows)

with open(possession_10min_csv_path, "w", newline="", encoding="utf-8") as f:
    fieldnames = ["window_index", "start_sec", "end_sec", "team_0_possession_pct", "team_1_possession_pct", "unknown_pct", "frames"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(window_rows)

if len(window_thirds_rows) > 0:
    with open(thirds_10min_csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(window_thirds_rows[0].keys()))
        writer.writeheader()
        writer.writerows(window_thirds_rows)

with open(my_team_overlay_csv_path, "w", newline="", encoding="utf-8") as f:
    fieldnames = ["frame_idx", "tracker_id", "team_id", "x", "y", "rx", "ry"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    if len(overlay_rows) > 0:
        writer.writerows(overlay_rows)

with open(events_csv_path, "w", newline="", encoding="utf-8") as f:
    if len(events) > 0:
        keys = sorted({k for row in events for k in row.keys()})
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(events)
    else:
        writer = csv.writer(f)
        writer.writerow(["info"])
        writer.writerow(["No pass/turnover events were detected with current thresholds."])

print("Stats saved to:", stats_dir)
print("-", summary_json_path)
print("-", my_team_thirds_json_path)
print("-", team_stats_csv_path)
print("-", team_thirds_csv_path)
print("-", player_stats_csv_path)
print("-", possession_10min_csv_path)
print("-", thirds_10min_csv_path)
print("-", events_csv_path)
print("-", my_team_overlay_csv_path)


In [ ]:
# CELL 13: Synced fast visual renderer (NO YOLO re-inference)
import os
import csv
import json
import glob
import cv2
import numpy as np
from tqdm import tqdm
import supervision as sv
from collections import defaultdict
from pathlib import Path

if "ensure_notebook_config" in globals():
    ensure_notebook_config(show_progress=False)

def _pick_stats_dir():
    candidates = []

    g_stats_dir = globals().get("stats_dir")
    if g_stats_dir:
        candidates.append(str(g_stats_dir))

    repo_root = str(globals().get("REPO_ROOT", "")).strip()
    if repo_root:
        candidates.append(os.path.join(repo_root, "examples", "soccer", "data", "stats"))

    cwd = str(Path.cwd().resolve())
    candidates.extend([
        os.path.join(cwd, "stats"),
        os.path.join(cwd, "data", "stats"),
        os.path.join(cwd, "examples", "soccer", "data", "stats"),
    ])

    seen = set()
    existing = []
    for c in candidates:
        if not c:
            continue
        p = os.path.abspath(str(c))
        if p in seen:
            continue
        seen.add(p)
        if os.path.isdir(p):
            existing.append(p)

    for p in existing:
        if os.path.exists(os.path.join(p, "summary.json")) or os.path.exists(os.path.join(p, "events.csv")):
            return p

    if existing:
        return existing[0]

    if repo_root:
        fallback = os.path.join(repo_root, "examples", "soccer", "data", "stats")
    else:
        fallback = os.path.join(cwd, "data", "stats")
    os.makedirs(fallback, exist_ok=True)
    return fallback


stats_dir = _pick_stats_dir()
events_csv_path = os.path.join(stats_dir, "events.csv")
overlay_csv_path = os.path.join(stats_dir, "my_team_overlay_tracks.csv")
summary_json_path = os.path.join(stats_dir, "summary.json")
output_video_path = os.path.join(stats_dir, "rendered_match_analysis.mp4")


def _resolve_source_video_path(source_video_path):
    candidates = []
    if source_video_path:
        candidates.append(str(source_video_path))

    g_source = globals().get("SOURCE_VIDEO_PATH")
    if g_source:
        candidates.append(str(g_source))

    if os.path.exists(summary_json_path):
        try:
            with open(summary_json_path, "r", encoding="utf-8") as f:
                summary_data = json.load(f)
            summary_source = summary_data.get("source_video_path")
            if summary_source:
                candidates.append(str(summary_source))
        except Exception:
            pass

    video_filename = str(globals().get("VIDEO_FILENAME", "")).strip()
    search_dirs = []
    data_dir = globals().get("DATA_DIR")
    if data_dir:
        search_dirs.append(str(data_dir))

    repo_root = str(globals().get("REPO_ROOT", "")).strip()
    if repo_root:
        search_dirs.append(os.path.join(repo_root, "examples", "soccer", "data"))

    stats_parent = os.path.dirname(stats_dir)
    if stats_parent:
        search_dirs.append(stats_parent)

    cwd = str(Path.cwd().resolve())
    search_dirs.extend([
        cwd,
        os.path.join(cwd, "data"),
        os.path.join(cwd, "examples", "soccer", "data"),
    ])

    seen = set()
    existing_dirs = []
    for d in search_dirs:
        if not d:
            continue
        p = os.path.abspath(str(d))
        if p in seen:
            continue
        seen.add(p)
        if os.path.isdir(p):
            existing_dirs.append(p)

    if video_filename:
        for d in existing_dirs:
            candidates.append(os.path.join(d, video_filename))

    for d in existing_dirs:
        for patt in ("*.mp4", "*.mov", "*.mkv"):
            matches = sorted(glob.glob(os.path.join(d, patt)))
            if matches:
                candidates.extend(matches)

    checked = []
    for c in candidates:
        if not c:
            continue
        p = os.path.abspath(str(c))
        if p in checked:
            continue
        checked.append(p)
        if os.path.exists(p):
            return p

    raise FileNotFoundError("Could not resolve source video for renderer. Set SOURCE_VIDEO_PATH to a valid file.")


def _load_events(csv_path):
    loaded = []
    if not os.path.exists(csv_path):
        return loaded
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if not row or "event" not in row:
                continue
            parsed = dict(row)
            for key in ("time_sec", "distance_px", "flight_time_sec"):
                if key in parsed and parsed[key] not in (None, ""):
                    try:
                        parsed[key] = float(parsed[key])
                    except Exception:
                        pass
            for key in ("team", "lost_by_team", "won_by_team"):
                if key in parsed and parsed[key] not in (None, ""):
                    try:
                        parsed[key] = int(float(parsed[key]))
                    except Exception:
                        pass
            loaded.append(parsed)
    return loaded


def _load_my_team_overlay_tracks(csv_path):
    tracks = defaultdict(list)
    current_team_id = int(globals().get("MY_TEAM_ID", 1))

    in_memory = globals().get("my_team_overlay_tracks")
    if isinstance(in_memory, dict) and len(in_memory) > 0:
        for frame_idx, rows in in_memory.items():
            try:
                frame_key = int(frame_idx)
            except Exception:
                continue
            for row in rows:
                if not isinstance(row, dict):
                    continue
                row_team_id = row.get("team_id", None)
                if row_team_id is None:
                    continue
                try:
                    row_team_id = int(row_team_id)
                except Exception:
                    continue
                if row_team_id != current_team_id:
                    continue
                tracks[frame_key].append(
                    {
                        "tracker_id": int(row["tracker_id"]),
                        "team_id": row_team_id,
                        "x": int(row["x"]),
                        "y": int(row["y"]),
                        "rx": int(row["rx"]),
                        "ry": int(row["ry"]),
                    }
                )
        if len(tracks) > 0:
            print(f"[Overlay] loaded from memory for MY_TEAM_ID={current_team_id}: {len(tracks)} frames")
            return tracks

    if not os.path.exists(csv_path):
        return tracks

    skipped_old_rows = 0
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if not row:
                continue
            if row.get("team_id", "") in (None, ""):
                skipped_old_rows += 1
                continue
            try:
                team_id = int(float(row["team_id"]))
            except Exception:
                continue
            if team_id != current_team_id:
                continue
            try:
                frame_idx = int(float(row["frame_idx"]))
                tracks[frame_idx].append(
                    {
                        "tracker_id": int(float(row["tracker_id"])),
                        "team_id": team_id,
                        "x": int(float(row["x"])),
                        "y": int(float(row["y"])),
                        "rx": int(float(row["rx"])),
                        "ry": int(float(row["ry"])),
                    }
                )
            except Exception:
                continue

    if skipped_old_rows > 0:
        print(
            f"[Overlay] skipped {skipped_old_rows} old overlay rows without team_id. "
            "Run Cell 12 once to regenerate filtered overlay tracks."
        )

    if len(tracks) > 0:
        print(f"[Overlay] loaded from CSV for MY_TEAM_ID={current_team_id}: {len(tracks)} frames")

    return tracks


def _draw_dashed_ellipse(frame, center, axes, color, thickness=2, dash_count=18, fill_alpha=0.18):
    overlay = frame.copy()
    cv2.ellipse(overlay, center, axes, 0, 0, 360, color, -1)
    cv2.addWeighted(overlay, fill_alpha, frame, 1.0 - fill_alpha, 0, frame)

    for i in range(dash_count):
        start_angle = int((360 / dash_count) * i)
        end_angle = int(start_angle + (360 / dash_count) * 0.55)
        cv2.ellipse(frame, center, axes, 0, start_angle, end_angle, color, thickness, lineType=cv2.LINE_AA)


def run_fast_rendering_synced(
    source_video_path=SOURCE_VIDEO_PATH,
    output_path=output_video_path,
    step=frame_step,
    draw_ball=True,
    draw_pass_text=True,
    draw_player_circles=False,
    overlay_color=(40, 210, 255),
    label_color=(15, 40, 40),
):
    source_video_path = _resolve_source_video_path(source_video_path)
    video_info = sv.VideoInfo.from_video_path(source_video_path)
    fps = float(video_info.fps) if video_info.fps else 25.0
    events = _load_events(events_csv_path)
    my_team_tracks = _load_my_team_overlay_tracks(overlay_csv_path)

    events_by_release_frame = defaultdict(list)
    for ev in events:
        t = float(ev.get("time_sec", 0.0))
        release_frame = int(round(t * fps))
        events_by_release_frame[release_frame].append(ev)

    max_frame_idx = max([0] + list(events_by_release_frame.keys()) + list(my_team_tracks.keys()))
    print(f"Rendering synced overlay to: {output_path}")
    print(f"- resolved source video   : {source_video_path}")
    print(f"- pass/interception events: {len(events)}")
    print(f"- my-team overlay frames   : {len(my_team_tracks)}")
    print(f"- overlay MY_TEAM_ID       : {int(globals().get('MY_TEAM_ID', 1))}")
    print(f"- max sampled frame idx    : {max_frame_idx}")

    generator = sv.get_video_frames_generator(source_path=source_video_path, stride=1)
    annotated_frames = []
    last_text = ""
    last_text_until = -1
    total_frames = int(video_info.total_frames) if video_info.total_frames else None

    for actual_frame_idx, frame in enumerate(tqdm(generator, total=total_frames, desc="Rendering synced video")):
        sampled_frame_idx = actual_frame_idx - (actual_frame_idx % step)

        if draw_player_circles:
            for row in my_team_tracks.get(sampled_frame_idx, []):
                row_team_id = row.get("team_id", None)
                if row_team_id is None or int(row_team_id) != int(globals().get("MY_TEAM_ID", 1)):
                    continue
                center = (int(row["x"]), int(row["y"]))
                axes = (max(10, int(row["rx"])), max(6, int(row["ry"])))
                _draw_dashed_ellipse(frame, center, axes, overlay_color, thickness=2, dash_count=18, fill_alpha=0.14)
                cv2.putText(
                    frame,
                    f"P-{int(row['tracker_id'])}",
                    (center[0] - 18, center[1] - max(16, axes[1] + 6)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.45,
                    label_color,
                    2,
                    cv2.LINE_AA,
                )

        release_events = events_by_release_frame.get(actual_frame_idx, [])
        if release_events:
            parts = []
            for ev in release_events:
                if ev.get("event") == "team_pass":
                    team = ev.get("team", "?")
                    kind = str(ev.get("pass_kind", "pass"))
                    dist = ev.get("distance_px", 0)
                    parts.append(f"PASS T{team} {kind} {dist}px")
                elif ev.get("event") == "team_interception":
                    lost_by = ev.get("lost_by_team", "?")
                    won_by = ev.get("won_by_team", "?")
                    parts.append(f"INTERCEPTION T{won_by} from T{lost_by}")
            if parts:
                last_text = " | ".join(parts)
                last_text_until = actual_frame_idx + int(round(fps * 1.6))

        if draw_pass_text and last_text and actual_frame_idx <= last_text_until:
            (text_w, text_h), _ = cv2.getTextSize(last_text, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2)
            pad = 10
            x1, y1 = 24, 24
            x2, y2 = x1 + text_w + 2 * pad, y1 + text_h + 2 * pad
            overlay = frame.copy()
            cv2.rectangle(overlay, (x1, y1), (x2, y2), (20, 20, 20), -1)
            cv2.addWeighted(overlay, 0.42, frame, 0.58, 0, frame)
            cv2.rectangle(frame, (x1, y1), (x2, y2), (40, 210, 255), 2)
            cv2.putText(
                frame,
                last_text,
                (x1 + pad, y2 - pad),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (255, 255, 255),
                2,
                cv2.LINE_AA,
            )

        annotated_frames.append(frame)

    print("Writing video...")
    try:
        sv.ImageSink.save_video(frames=annotated_frames, output_path=output_path, fps=video_info.fps)
    except AttributeError:
        print("[Renderer] ImageSink.save_video is unavailable in this supervision version. Falling back to OpenCV writer.")
        if len(annotated_frames) == 0:
            raise RuntimeError("No frames were produced to write.")

        first = annotated_frames[0]
        h, w = first.shape[:2]
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        out = cv2.VideoWriter(output_path, fourcc, float(video_info.fps) if video_info.fps else 25.0, (w, h))
        if not out.isOpened():
            raise RuntimeError(f"Failed to open VideoWriter for output: {output_path}")
        try:
            for fr in annotated_frames:
                if fr.dtype != np.uint8:
                    fr = np.clip(fr, 0, 255).astype(np.uint8)
                if fr.shape[0] != h or fr.shape[1] != w:
                    fr = cv2.resize(fr, (w, h), interpolation=cv2.INTER_LINEAR)
                out.write(fr)
        finally:
            out.release()

    print("Saved:", output_path)
    return output_path


rendered_path = run_fast_rendering_synced()
globals()["rendered_match_analysis_path"] = rendered_path
print("Rendered video path:", rendered_path)


In [ ]:
#CELL 13      Visual analytics for thirds possession (RunPod/Colab compatible)
import os
import json
import subprocess
import sys
import matplotlib.pyplot as plt

try:
    import pandas as pd
except ModuleNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas"], check=False)
    import pandas as pd

stats_dir = os.path.join(REPO_ROOT, "examples", "soccer", "data", "stats")
team_thirds_path = os.path.join(stats_dir, "team_possession_by_third.csv")
thirds_10min_path = os.path.join(stats_dir, "possession_by_third_10min.csv")
my_team_path = os.path.join(stats_dir, "my_team_thirds_summary.json")

for p in [team_thirds_path, thirds_10min_path, my_team_path]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing {p}. Run the stats cell first.")

team_thirds_df = pd.read_csv(team_thirds_path)
thirds_10_df = pd.read_csv(thirds_10min_path)
with open(my_team_path, "r", encoding="utf-8") as f:
    my_team_thirds = json.load(f)

team_thirds_df["team_label"] = team_thirds_df["team_id"].apply(
    lambda t: "My Team" if int(t) == int(MY_TEAM_ID) else "Opponent"
)

print("My team thirds summary:")
display(pd.DataFrame([my_team_thirds]))
print("\nTeam-level thirds table:")
display(team_thirds_df[[
    "team_id", "team_label", "goal_side",
    "defensive_sec", "middle_sec", "offensive_sec",
    "defensive_pct_of_team_possession", "middle_pct_of_team_possession", "offensive_pct_of_team_possession"
]])

fig1, ax1 = plt.subplots(figsize=(10, 5))
plot_df = team_thirds_df.set_index("team_label")[["defensive_sec", "middle_sec", "offensive_sec"]]
plot_df.plot(kind="bar", stacked=True, ax=ax1, color=["#1f77b4", "#ffbf00", "#d62728"] )
ax1.set_title("Possession Time by Third (Seconds)")
ax1.set_ylabel("Seconds")
ax1.set_xlabel("")
ax1.legend(title="Third")
plt.xticks(rotation=0)
plt.tight_layout()

my_10_df = thirds_10_df[thirds_10_df["team_id"] == int(MY_TEAM_ID)].copy()
my_10_df = my_10_df.sort_values(["window_index"])
my_10_df["window_label"] = my_10_df.apply(
    lambda r: f"{int(r['start_sec']//60)}-{int(r['end_sec']//60)} min", axis=1
)

fig2, ax2 = plt.subplots(figsize=(12, 5))
my_10_plot = my_10_df.set_index("window_label")[["defensive_sec", "middle_sec", "offensive_sec"]]
my_10_plot.plot(kind="bar", stacked=True, ax=ax2, color=["#1f77b4", "#ffbf00", "#d62728"] )
ax2.set_title("My Team Possession by Third (Per 10-Minute Window)")
ax2.set_ylabel("Seconds")
ax2.set_xlabel("Window")
ax2.legend(title="Third")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()

chart_team_path = os.path.join(stats_dir, "chart_team_thirds_seconds.png")
chart_my10_path = os.path.join(stats_dir, "chart_my_team_thirds_10min.png")
fig1.savefig(chart_team_path, dpi=150, bbox_inches="tight")
fig2.savefig(chart_my10_path, dpi=150, bbox_inches="tight")
plt.show()

print("Saved charts:")
print("-", chart_team_path)
print("-", chart_my10_path)

In [ ]:
# CELL: Distance between our goalkeeper and nearest defender (per minute, up to 90 minutes)
import os
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import supervision as sv
from tqdm import tqdm
from ultralytics import YOLO

if "ensure_notebook_config" in globals():
    ensure_notebook_config(show_progress=False)

if "soccer_main" not in globals():
    import sys
    if REPO_ROOT not in sys.path:
        sys.path.insert(0, REPO_ROOT)
    os.chdir(os.path.join(REPO_ROOT, "examples", "soccer"))
    import main as soccer_main

stats_dir = os.path.join(REPO_ROOT, "examples", "soccer", "data", "stats")
os.makedirs(stats_dir, exist_ok=True)

source_video_path = SOURCE_VIDEO_PATH
video_info = sv.VideoInfo.from_video_path(source_video_path)
fps = float(video_info.fps) if video_info.fps else 25.0

# Keep it aligned with stats extraction speed profile
frame_step = max(3, int(globals().get("STATS_FRAME_STRIDE", 3)))
infer_batch = max(1, int(globals().get("INFER_BATCH_SIZE", 32)))
player_imgsz = min(640, int(globals().get("PLAYER_DET_IMGSZ", 640)))
my_team_id = int(globals().get("MY_TEAM_ID", 1))
my_goal_side = str(globals().get("MY_TEAM_OWN_GOAL_SIDE", "right")).strip().lower()

if my_team_id not in (0, 1):
    raise ValueError("MY_TEAM_ID must be 0 or 1")
if my_goal_side not in ("left", "right"):
    raise ValueError("MY_TEAM_OWN_GOAL_SIDE must be left or right")

player_model_local = globals().get("player_model")
if player_model_local is None:
    player_model_local = YOLO(soccer_main.PLAYER_DETECTION_MODEL_PATH).to(device=globals().get("DEVICE", "cuda"))

team_classifier_local = globals().get("team_classifier")
if team_classifier_local is None:
    # Fallback fit if Cell 12 was not executed
    fit_stride = max(10, frame_step)
    fit_cap = 1200
    fit_crops = []
    fit_gen = sv.get_video_frames_generator(source_path=source_video_path, stride=fit_stride)
    batch = []
    for frame in fit_gen:
        batch.append(frame)
        if len(batch) == infer_batch:
            results = player_model_local(batch, imgsz=player_imgsz, half=bool(globals().get("USE_FP16", True)), verbose=False)
            for fr, res in zip(batch, results):
                det = sv.Detections.from_ultralytics(res)
                players_only = det[det.class_id == soccer_main.PLAYER_CLASS_ID]
                fit_crops.extend(soccer_main.get_crops(fr, players_only))
                if len(fit_crops) >= fit_cap:
                    break
            batch = []
            if len(fit_crops) >= fit_cap:
                break
    if len(batch) > 0 and len(fit_crops) < fit_cap:
        results = player_model_local(batch, imgsz=player_imgsz, half=bool(globals().get("USE_FP16", True)), verbose=False)
        for fr, res in zip(batch, results):
            det = sv.Detections.from_ultralytics(res)
            players_only = det[det.class_id == soccer_main.PLAYER_CLASS_ID]
            fit_crops.extend(soccer_main.get_crops(fr, players_only))
            if len(fit_crops) >= fit_cap:
                break

    if len(fit_crops) < 40:
        raise RuntimeError("Not enough crops to fit team classifier for GK-defender distance analysis")

    team_classifier_local = soccer_main.TeamClassifier(device=globals().get("DEVICE", "cuda"))
    team_classifier_local.fit(fit_crops)

def _safe_goalkeeper_team(players_det, players_team_id, goalkeepers_det):
    if len(goalkeepers_det) == 0:
        return np.array([], dtype=int)
    if len(players_det) == 0 or len(players_team_id) == 0:
        return np.full(len(goalkeepers_det), -1, dtype=int)

    valid = players_team_id[np.isin(players_team_id, np.array([0, 1]))]
    if len(valid) == 0:
        return np.full(len(goalkeepers_det), -1, dtype=int)

    unique_valid = np.unique(valid)
    if len(unique_valid) < 2:
        return np.full(len(goalkeepers_det), int(unique_valid[0]), dtype=int)

    majority = int(np.bincount(valid).argmax())
    tmp_team_id = players_team_id.copy()
    tmp_team_id[tmp_team_id == -1] = majority
    return soccer_main.resolve_goalkeepers_team_id(players_det, tmp_team_id, goalkeepers_det).astype(int)

minutes_distances = {m: [] for m in range(1, 91)}

def _choose_our_gk(gk_xy, goalkeepers_team_id, side):
    idx = [i for i, t in enumerate(goalkeepers_team_id) if int(t) == my_team_id]
    if not idx:
        return None
    if len(idx) == 1:
        return gk_xy[idx[0]]
    # If more than one candidate appears, pick the one closest to our own goal side
    if side == "left":
        best_i = min(idx, key=lambda i: float(gk_xy[i][0]))
    else:
        best_i = max(idx, key=lambda i: float(gk_xy[i][0]))
    return gk_xy[best_i]

frame_gen = sv.get_video_frames_generator(source_path=source_video_path, stride=frame_step)
batch_frames = []
processed_sampled_frames = 0
for frame in tqdm(frame_gen, desc="GK-Defender distance (sampled frames)"):
    batch_frames.append(frame)
    if len(batch_frames) < infer_batch:
        continue

    results = player_model_local(batch_frames, imgsz=player_imgsz, half=bool(globals().get("USE_FP16", True)), verbose=False)
    for j, (fr, res) in enumerate(zip(batch_frames, results)):
        sampled_frame_idx = (processed_sampled_frames + j) * frame_step
        time_sec = sampled_frame_idx / fps if fps > 0 else 0.0
        minute_num = int(time_sec // 60) + 1
        if minute_num > 90:
            continue

        det = sv.Detections.from_ultralytics(res)
        players = det[det.class_id == soccer_main.PLAYER_CLASS_ID]
        goalkeepers = det[det.class_id == soccer_main.GOALKEEPER_CLASS_ID]

        if len(players) == 0 or len(goalkeepers) == 0:
            continue

        player_crops = soccer_main.get_crops(fr, players)
        if len(player_crops) == 0:
            continue

        players_team_id = team_classifier_local.predict(player_crops).astype(int)
        gk_team_id = _safe_goalkeeper_team(players, players_team_id, goalkeepers)
        if len(gk_team_id) == 0:
            continue

        players_xy = players.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
        gk_xy = goalkeepers.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)

        our_gk_xy = _choose_our_gk(gk_xy, gk_team_id, my_goal_side)
        if our_gk_xy is None:
            continue

        # Defenders: nearest field player from our team to our goalkeeper
        our_players_idx = np.where(players_team_id == my_team_id)[0]
        if len(our_players_idx) == 0:
            continue

        our_players_xy = players_xy[our_players_idx]
        dists = np.linalg.norm(our_players_xy - our_gk_xy, axis=1)
        if len(dists) == 0:
            continue

        nearest_defender_dist = float(np.min(dists))
        minutes_distances[minute_num].append(nearest_defender_dist)

    processed_sampled_frames += len(batch_frames)
    batch_frames = []

if len(batch_frames) > 0:
    results = player_model_local(batch_frames, imgsz=player_imgsz, half=bool(globals().get("USE_FP16", True)), verbose=False)
    for j, (fr, res) in enumerate(zip(batch_frames, results)):
        sampled_frame_idx = (processed_sampled_frames + j) * frame_step
        time_sec = sampled_frame_idx / fps if fps > 0 else 0.0
        minute_num = int(time_sec // 60) + 1
        if minute_num > 90:
            continue

        det = sv.Detections.from_ultralytics(res)
        players = det[det.class_id == soccer_main.PLAYER_CLASS_ID]
        goalkeepers = det[det.class_id == soccer_main.GOALKEEPER_CLASS_ID]

        if len(players) == 0 or len(goalkeepers) == 0:
            continue

        player_crops = soccer_main.get_crops(fr, players)
        if len(player_crops) == 0:
            continue

        players_team_id = team_classifier_local.predict(player_crops).astype(int)
        gk_team_id = _safe_goalkeeper_team(players, players_team_id, goalkeepers)
        if len(gk_team_id) == 0:
            continue

        players_xy = players.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
        gk_xy = goalkeepers.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)

        our_gk_xy = _choose_our_gk(gk_xy, gk_team_id, my_goal_side)
        if our_gk_xy is None:
            continue

        our_players_idx = np.where(players_team_id == my_team_id)[0]
        if len(our_players_idx) == 0:
            continue

        our_players_xy = players_xy[our_players_idx]
        dists = np.linalg.norm(our_players_xy - our_gk_xy, axis=1)
        if len(dists) == 0:
            continue

        nearest_defender_dist = float(np.min(dists))
        minutes_distances[minute_num].append(nearest_defender_dist)

rows = []
for m in range(1, 91):
    vals = minutes_distances[m]
    rows.append(
        {
            "minute": m,
            "nearest_defender_dist_px_mean": round(float(np.mean(vals)), 2) if len(vals) > 0 else np.nan,
            "nearest_defender_dist_px_median": round(float(np.median(vals)), 2) if len(vals) > 0 else np.nan,
            "samples": int(len(vals)),
        }
    )

df_gk_def = pd.DataFrame(rows)
csv_path = os.path.join(stats_dir, "gk_nearest_defender_distance_by_minute.csv")
png_path = os.path.join(stats_dir, "gk_nearest_defender_distance_by_minute.png")
df_gk_def.to_csv(csv_path, index=False, encoding="utf-8")

plt.figure(figsize=(14, 5))
plt.plot(df_gk_def["minute"], df_gk_def["nearest_defender_dist_px_mean"], linewidth=2, label="Mean distance")
plt.plot(df_gk_def["minute"], df_gk_def["nearest_defender_dist_px_median"], linewidth=1.8, linestyle="--", label="Median distance")
plt.xlim(1, 90)
plt.xticks(np.arange(1, 91, 5))
plt.xlabel("Minute")
plt.ylabel("Distance (pixels)")
plt.title("Our Goalkeeper vs Nearest Defender Distance by Minute (1-90)")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(png_path, dpi=150, bbox_inches="tight")
plt.show()

print("Saved:")
print("-", csv_path)
print("-", png_path)
display(df_gk_def.head(15))

In [ ]:
#CELL 14       Export result video (Colab + RunPod)
import os

if not os.path.exists(TARGET_VIDEO_PATH):
    raise FileNotFoundError(f"Result video not found: {TARGET_VIDEO_PATH}")

print("Result video:", TARGET_VIDEO_PATH)
try:
    from google.colab import files as colab_files  # type: ignore
    colab_files.download(TARGET_VIDEO_PATH)
except Exception:
    print("Not in Colab. Download file directly from path above.")

In [ ]:
#CELL 15     Export stats ZIP (Colab + RunPod)
import os
import shutil

stats_zip_path = os.path.join(stats_dir, "stats_bundle.zip")
if os.path.exists(stats_zip_path):
    os.remove(stats_zip_path)

shutil.make_archive(stats_zip_path.replace('.zip', ''), 'zip', stats_dir)
print("Stats ZIP:", stats_zip_path)

try:
    from google.colab import files as colab_files  # type: ignore
    colab_files.download(stats_zip_path)
except Exception:
    print("Not in Colab. Download file directly from path above.")